# `needle.api` walkthrough

This notebook exercises the public `needle.api` surface end to end, outside of a running
`law`/`b2luigi` process:

1. **`load_config`** — load and resolve a Hydra config
2. **`run`** — submit the DAG to the `b2luigi` backend (in-process)
3. **`load_snapshot`** / **`Estimator`** — load trained checkpoints and run aggregated inference
4. **`aggregate_siblings`** — call the aggregation logic directly, including a custom aggregator
5. **`train_single`** — train one estimator directly, bypassing the DAG fan-out
6. **`init`** — scaffold a new NEEDLE project
7. **`configure_law`** / **`configure_b2luigi`** — configure a backend from Python instead of `setup.sh`/`settings.json`

It uses `tests/conf_tests/config.yaml` (a tiny two-estimator DAG: `model_A`, and `model_B` which
`requires: ["model_A"]`) against the small parquet file bundled at
`examples/fair_universe_demo/test_data/`, so it runs standalone with no external data or
environment variables required. All run output is written to a temporary directory, so re-running
this notebook never leaves artifacts in the repo.


In [1]:
import tempfile
from pathlib import Path

import omegaconf
import torch

import needle.api as api

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "tests" / "conf_tests" / "config.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent  # running from examples/

CONFIG_PATH = REPO_ROOT / "tests" / "conf_tests" / "config.yaml"
DEMO_PARQUET = REPO_ROOT / "examples" / "fair_universe_demo" / "test_data" / "FAIR_Universe_HiggsML_data.parquet"
assert CONFIG_PATH.exists(), CONFIG_PATH
assert DEMO_PARQUET.exists(), DEMO_PARQUET

work_dir = Path(tempfile.mkdtemp(prefix="needle_api_demo_"))
print("Scratch results dir:", work_dir)


Scratch results dir: /tmp/needle_api_demo_kt8xbq13


## 1. `load_config`

Loads and fully resolves a Hydra config (defaults applied, `*_override` fields populated, DAG
validated) without needing a running `law`/`b2luigi` process. Returns the resolved
`needle.utils.config_schema.MainConfig` as an OmegaConf `DictConfig`.


In [2]:
cfg = api.load_config(CONFIG_PATH)

print("Estimators:", list(cfg.estimators.keys()))
print("model_A requires:", cfg.estimators.model_A.requires)
print("model_B requires:", cfg.estimators.model_B.requires)
print("model_A systematics:", list(cfg.estimators.model_A.expands.systematics.keys()))
print("model_A folds/ensembles:", cfg.estimators.model_A.expands.folds, cfg.estimators.model_A.expands.ensembles)


Estimators: ['model_A', 'model_B']
model_A requires: None
model_B requires: ['model_A']
model_A systematics: ['nominal', 'up_qcd']
model_A folds/ensembles: {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}} {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}}


In [3]:
# Hydra override strings work the same way as `--hydra-overrides` on the CLI.
cfg_overridden = api.load_config(CONFIG_PATH, overrides=["estimators.model_A.expands.folds=3"])
print("Default folds:", cfg.estimators.model_A.expands.folds)
print("Overridden folds:", cfg_overridden.estimators.model_A.expands.folds)


Default folds: {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}}
Overridden folds: {'num': 3, 'aggregation': {'method': 'mean', 'metric_key': None}}


## 2. `run` — submit the DAG to the `b2luigi` backend

`run()` submits any task by class name to either backend. Here we point both estimators' dataset
paths at the bundled demo parquet, save the config, and run `MainTask` in-process via the
`b2luigi` backend (the default) — this trains every estimator in the DAG (`model_A` and, because
it `requires: ["model_A"]`, `model_B`), then writes `config.yaml` and `dag_snapshot.json` into
`results_path`.


In [4]:
run_cfg = api.load_config(CONFIG_PATH)
for name in run_cfg.estimators:
    run_cfg.estimators[name].dataset_override.paths = str(DEMO_PARQUET)
run_cfg._resolved = True

run_config_file = work_dir / "config.yaml"
omegaconf.OmegaConf.save(run_cfg, run_config_file, resolve=True)

result = api.run(
    task="MainTask",
    backend="b2luigi",
    config_file=run_config_file,
    results_path=str(work_dir),
)
print(result)


INFO: NEEDLE-api.run (11:35:14) - Running with `b2luigi` workflow backend


INFO: Informed scheduler that task   b2luigi.MainTask__99914b932b   has status   PENDING


INFO: Informed scheduler that task   b2luigi.EstimatorTask_model_B_8a0f906e12   has status   PENDING


INFO: Informed scheduler that task   b2luigi.SystematicTask_model_B_nominal_43ba5637db   has status   PENDING


INFO: Informed scheduler that task   b2luigi.EnsembleTask_0_model_B_nominal_3c5493ea1d   has status   PENDING


...

INFO: Worker Worker(salt=619384331, workers=1, host=portal2, username=kschmidt, pid=2571621) was stopped. Shutting down Keep-Alive thread


INFO:luigi-interface:Worker Worker(salt=619384331, workers=1, host=portal2, username=kschmidt, pid=2571621) was stopped. Shutting down Keep-Alive thread


INFO: 
===== Luigi Execution Summary =====

Scheduled 18 tasks of which:
* 18 ran successfully:
    - 3 b2luigi.EnsembleTask(...)
    - 2 b2luigi.EstimatorTask(...)
    - 5 b2luigi.FoldTask(...)
    - 1 b2luigi.MainTask()
    - 2 b2luigi.SystematicTask(...)
    ...

This progress looks :) because there were no failed tasks or missing dependencies

===== Luigi Execution Summary =====



INFO:luigi-interface:
===== Luigi Execution Summary =====

Scheduled 18 tasks of which:
* 18 ran successfully:
    - 3 b2luigi.EnsembleTask(...)
    - 2 b2luigi.EstimatorTask(...)
    - 5 b2luigi.FoldTask(...)
    - 1 b2luigi.MainTask()
    - 2 b2luigi.SystematicTask(...)
    ...

This progress looks :) because there were no failed tasks or missing dependencies

===== Luigi Execution Summary =====



RunResult(returncode=None)


In [5]:
cached_config = work_dir / "config.yaml"
snapshot_file = work_dir / "dag_snapshot.json"
print("Cached config written:", cached_config.exists())
print("DAG snapshot written:", snapshot_file.exists())
print(snapshot_file.read_text()[:400], "...")


Cached config written: True
DAG snapshot written: True
{
    "est=model_A&syst=up_qcd&ensem=0&fold=0": "/tmp/needle_api_demo_kt8xbq13/est__model_A/syst__up_qcd/ensem__0/fold__0/model.ckpt",
    "est=model_A&syst=up_qcd&ensem=0&fold=1": "/tmp/needle_api_demo_kt8xbq13/est__model_A/syst__up_qcd/ensem__0/fold__1/model.ckpt",
    "est=model_A&syst=up_qcd&ensem=1&fold=0": "/tmp/needle_api_demo_kt8xbq13/est__model_A/syst__up_qcd/ensem__1/fold__0/model.ckpt", ...


## 3. `load_snapshot` and `Estimator` — load checkpoints and run inference

`load_snapshot` unflattens `dag_snapshot.json` into `{systematic: {ensemble: {fold: ckpt_path}}}`
for one estimator. `Estimator` wraps that: it loads every checkpoint, then aggregates bottom-up
(folds → ensembles → systematics) using each level's `AggregationSpec` from the estimator's
config, producing a single `(mean, std)` prediction.


In [6]:
snapshot = api.load_snapshot(work_dir, "model_A")
print("Systematics trained for model_A:", list(snapshot.keys()))
for systematic, ensembles in snapshot.items():
    for ensemble, folds in ensembles.items():
        print(f"  {systematic=} {ensemble=} folds={sorted(folds.keys())}")


Systematics trained for model_A: ['up_qcd']
  systematic='up_qcd' ensemble=0 folds=[0, 1]
  systematic='up_qcd' ensemble=1 folds=[0, 1]


In [7]:
model = api.Estimator(work_dir, "model_A")
print(f"Loaded {len(model.models)} checkpoints "
      f"({len(snapshot)} systematic(s) x 2 ensembles x 2 folds = 4)")

n_features = len(run_cfg.estimators.model_A.dataset_override.features_columns)
x = torch.rand(6, n_features)
mean, std = model(x)
print("mean.shape:", mean.shape, " std.shape:", std.shape)
mean, std


Loaded 4 checkpoints (1 systematic(s) x 2 ensembles x 2 folds = 4)
mean.shape: torch.Size([6, 1])  std.shape: torch.Size([6, 1])


(tensor([[-0.2717],
         [-0.3202],
         [-0.3607],
         [-0.3396],
         [-0.3443],
         [-0.3556]]),
 tensor([[0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.]]))

## 4. `aggregate_siblings` — direct use, including a custom aggregator

`Estimator` calls `aggregate_siblings` internally, but it's also usable standalone: pass a list of
sibling prediction tensors and an `AggregationSpec`. Built-in methods are `"mean"`, `"sum"`,
`"best"` (needs `metrics=`); anything else is resolved as a dotted import path to a user-supplied
callable — there's no generic `weights` field in `AggregationSpec` on purpose, so a callable that
wants weights just captures them itself. `tests/api/test_eval.py::_weighted_mean` is a small
worked example (also documented in `docs/concepts/hydra_config.md`).


In [8]:
from needle.utils.config_schema import AggregationSpec

outputs = [torch.zeros(2, 1), torch.full((2, 1), 4.0)]

mean_agg, std_agg = api.aggregate_siblings(outputs, AggregationSpec(method="mean"))
print("mean:", mean_agg.squeeze().tolist())

best_agg, _ = api.aggregate_siblings(outputs, AggregationSpec(method="best"), metrics=[0.9, 0.1])
print("best (lowest metric wins):", best_agg.squeeze().tolist())

# Dotted path to a user-supplied aggregator (weights: 3:1 in favor of the first sibling).
custom_agg, custom_std = api.aggregate_siblings(
    outputs, AggregationSpec(method="tests.api.test_eval._weighted_mean")
)
print("custom weighted mean:", custom_agg.squeeze().tolist())


mean: [2.0, 2.0]
best (lowest metric wins): [4.0, 4.0]
custom weighted mean: [1.0, 1.0]


## 5. `train_single` — train one estimator directly

Bypasses `requires`/systematics/ensembles/folds fan-out entirely and trains exactly one
`TrainingTask(single=True)`. Useful for quick iteration on a single model. `b2luigi` only (`law`
has no in-process execution path).


In [9]:
single_cfg = api.load_config(CONFIG_PATH)
single_cfg.estimators.model_A.dataset_override.paths = str(DEMO_PARQUET)
single_cfg._resolved = True

single_config_file = work_dir / "single_config.yaml"
omegaconf.OmegaConf.save(single_cfg, single_config_file, resolve=True)

single_results = work_dir / "single"
api.train_single(
    estimator="model_A",
    config_file=str(single_config_file),
    results_path=str(single_results),
)
print(sorted(p.name for p in single_results.rglob("*.ckpt")))


/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


INFO: NEEDLE-etl (11:36:00) - Loaded 1000 events with 1 column(s): ['PRI_lep_pt']


INFO: NEEDLE-etl (11:36:00) - Loaded 1000 events with 1 column(s): ['PRI_n_jets']


...

Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 27.01it/s, v_num=f9c6]

2026/09/23 11:36:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/09/23 11:36:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


INFO: NEEDLE-training (11:36:07) - Successfully completed training for Estimator 'model_A'


['epoch=0-step=1.ckpt', 'model.ckpt', 'model.ckpt']


## 6. `init` — scaffold a new NEEDLE project

`needle init` (and its Python equivalent `api.init`) writes `law.cfg`, `settings.json`,
`setup.sh` and a default `conf/` tree for a new project. Run here into a throwaway directory so it
doesn't touch this repo.


In [10]:
with tempfile.TemporaryDirectory(prefix="needle_init_demo_") as scaffold_dir:
    scaffold_result = api.init(scaffold_dir, backend="both")
    print(scaffold_result)
    print(sorted(p.relative_to(scaffold_dir) for p in Path(scaffold_dir).rglob("*") if p.is_file()))


Created 'law.cfg' (LAW config file for managing Tasks)
Created 'index' (Index of needle.law_tasks, update with `law index`)
Created 'settings.json' (b2luigi settings file)
Created 'tasks.py' (b2luigi task index, required by the `b2luigi` CLI (incl. batch workers))
Created 'setup.sh' (Setup script for setting up the NEEDLE environment)
Created 'conf' (Config directory following the hydra schema)
InitResult(created=[PosixPath('/tmp/needle_init_demo_u0ze65vs/law.cfg'), PosixPath('/tmp/needle_init_demo_u0ze65vs/index'), PosixPath('/tmp/needle_init_demo_u0ze65vs/settings.json'), PosixPath('/tmp/needle_init_demo_u0ze65vs/tasks.py'), PosixPath('/tmp/needle_init_demo_u0ze65vs/setup.sh'), PosixPath('/tmp/needle_init_demo_u0ze65vs/conf')], skipped=[])
[PosixPath('conf/config.yaml'), PosixPath('conf/datamodules/padded.yaml'), PosixPath('conf/datamodules/pandas.yaml'), PosixPath('conf/datasets/default.yaml'), PosixPath('conf/example_config_overfull.yaml'), PosixPath('conf/models/mock_transformer.y

## 7. `configure_law` / `configure_b2luigi`

Both set backend configuration from Python instead of sourcing `setup.sh` (`law`) or editing
`settings.json` (`b2luigi`) — handy in a notebook or script where sourcing a shell file isn't an
option.

```python
# law: sets LAW_HOME / LAW_CONFIG_FILE for this process.
api.configure_law(law_home=str(REPO_ROOT / ".law"), law_config_file=str(REPO_ROOT / "law.cfg"))

# b2luigi: sets batch-system settings programmatically; `run(backend="b2luigi", ...)` calls this
# internally with `batch_system=`, so most callers never need it directly.
api.configure_b2luigi(batch_system="local")
```

Not executed here since `run(backend="b2luigi", ...)` above already configures it internally, and
the `law` backend shells out to the `law` CLI, which needs a one-time `law index` against a real
project — out of scope for this self-contained demo.
